<a href="https://colab.research.google.com/github/franfgv9/RI/blob/main/practica1b_indice_invertido_disco.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Resumen

En esta práctica, aplicaremos los conocimientos adquiridos en las clases de teoría y prácticas anteriores para crear un índice invertido considerando restricciones impuestas por el hardware subyacente a la hora de construirlo. En concreto, las tareas a realizar en esta práctica son:

1. **Construir un índice usando el algoritmo *naïve* y ser capaz de guardarlo/cargarlo de disco**. Usaremos el mismo corpus de documentos que en la práctica anterior.


# Requisitos para la construcción del índice

In [ ]:
# Añade aquí los imports que necesites
import sys
import pickle as pkl
import array
import os
import timeit
import contextlib
from pathlib import Path
from urllib.parse import urlparse
import urllib.request
import zipfile

## Límite de memoria disponible

Para comprobar que no rebasas los límites de uso de memoria pre-establecidos para realizar la construcción del índice de la manera adecuada, debes ejecutar este notebook lanzando `jupyter-notebook` desde un terminal del *shell*, en el que previamente hayas ejecutado la siguiente orden:

```bash
$ ulimit -v $(echo $(( 1024 * 1024 )))
```

In [ ]:
! ulimit -v $(echo $(( 1024 * 1024 )))

In [ ]:
!ulimit -a | grep "virtual memory"

virtual memory              (kbytes, -v) unlimited


Si has establecido correctamente los límites de uso de memoria del proceso `python3` que está ejecutando este *notebook*, el resultado de ejecutar la orden anterior debería ser:
```
virtual memory              (kbytes, -v) 1048576
```

Trabajaremos en esta práctica con el mismo corpus de documentos que en la anterior. Ejecuta la siguiente celda para obtenerlo en caso de que no lo tengas ya descargado en el mismo directorio del notebook.

## Corpus de documentos. Directorios de entrada y salida

In [ ]:
# URL del corpus de documentos
data_url = 'https://ditec.um.es/~lfmaimo/docencia/ri/practica1-datos.zip'

local_filename = os.path.basename(urlparse(data_url).path)   # urlparse().path se queda con la ruta y os.path.basename() se queda con el nombre dle ficher final
if not Path(local_filename).exists():
    print(f'Descargando {data_url} a {local_filename}...')
    urllib.request.urlretrieve(data_url, local_filename)

# fichero ZIP descargado → abrir → extraer su contenido → cerrar
zip_ref = zipfile.ZipFile(local_filename, 'r')
zip_ref.extractall()
zip_ref.close()

A continuación se definen las rutas relativas a los directorios que utilizaremos en el notebook tanto para generar los ficheros resultantes del indizado, como los directorios en los que se ubican los ficheros de entrada (corpus, tests, etc.)

In [ ]:
# Directorios de entrada y salida (generados)
corpus_dir = 'data/corpus'
toy_dir = 'data/toy'            # corpus pqueños de prueba
out_dir_naive='output/naive/index'          # salida del índice naïve construido sobre el corpus
out_dir_naive_toy='output/naive/toy_index'

# Cambia esto a "True" para indexar el corpus íntegro en memoria (requiere más de 1GiB de memoria)
build_naive_index_full = False

terms_filename = 'terms.dict'
docs_filename = 'docs.dict'
postings_index_filename = 'postings.index'          # fichero donde acabarán almacenándose las postings lists
postings_metadata_filename = 'postings_metadata.dict'           # metadatos necesarios para trabajar posteriormente con esas postings almacenadas

Ejecuta la siguiente celda para asegurarte de que todo está en su sitio:

In [ ]:
# Comprobar que determinados directorios que deben existir previamente realmente están disponibles

from pathlib import Path
import os

# Check if all directories exist
def check_directories(*directories):
    for path in directories:
        directory = Path(path)
        if not directory.is_dir():
            raise FileNotFoundError(f"Cannot find required directory at path '{path}'.")
        else:
            print(f"Found required directory at '{path}'.")

# Create output directories
def create_directories(*directories):
    for path in directories:
        try:
            os.makedirs(path, exist_ok=False)
            print(f"Output directory '{path}' has been created.")
        except FileExistsError:
            print(f"Output directory '{path}' already exists.")
            pass
        except Exception as e:
            print(f"Could not create directory at path '{path}'. Error: {e}")

check_directories(corpus_dir, toy_dir)
create_directories(out_dir_naive, out_dir_naive_toy)


Found required directory at 'data/corpus'.
Found required directory at 'data/toy'.
Output directory 'output/naive/index' already exists.
Output directory 'output/naive/toy_index' already exists.


El índice se almacenará en el directorio indicado por la variable `out_dir_naive`. También utilizaremos un directorio, dado por el valor de `out_dir_naive_toy`, donde se generará un índice de prueba a partir de datos de juguete. Por su parte, `tmp_dir` indica la ruta al directorio en el que se guardarán algunos archivos temporales para los índices de juguete.

## La clase auxiliar *IdMap*

Para hacer más eficiente la construcción de índices, representaremos los términos del vocabulario como *termIDs* (en lugar de cadenas), donde cada *termID* es un número de serie único. Podemos construir el mapeo de términos a termIDs sobre la marcha mientras procesamos la colección. Del mismo modo, también representamos los documentos como *docIDs* (en lugar de cadenas).

Con el fin de simplificar la correspondencia entre cadenas e ids numéricos, debes programar una clase auxiliar llamada `IdMap`. Esta clase se encargará de asignar biyectivamente término a termID y doc a docID.

Para esto, la clase deberá contener dos atributos:
* `str_to_id`: Un diccionario que mapeará cada cadena a su id numéricos, permitiendo un acceso eficiente al id numérico a partir de la cadena.
* `id_to_str`: Una lista para asociar cada id numérico (posición en la lista) a su correspondiente cadena de caracteres, permitiendo un acceso y almacenamiento eficientes de ids numéricos a cadenas.


A la vista de estos requisitos, programa las funciones `_get_str` y `_get_id` en el siguiente código. La única interfaz a esta clase es proporcionada por `__getitem__` que obtiene el mapeo correcto dependiendo del tipo de clave. Como ya sabrás, los métodos que comienzan por _ son **métodos internos** y no deberían ser llamados desde fuera de la clase.

**Documentación recomendada**: *Si lo necesitas, consulta [la documentación sobre métodos especiales (o "mágicos")](https://docs.python.org/3.7/reference/datamodel.html#special-method-names) de Python, tal como como `__getitem__` ). También puede resultarte útil este [breve tutorial](https://www.omkarpathak.in/2018/04/11/python-getitem-and-setitem/).*

In [ ]:
class IdMap:
    """Clase auxliar para mapear cadenas a identificadores."""

    def __init__(self):
        self.str_to_id = {}      # como clave la cadena str y como valor la pos de la posting list, es decir, el term_id o doc_id   --> {"fran":0, "juan":1, "hola":2}
        self.id_to_str = []       # aqui interesa solo la posición --> en cada posicion ordenada tenemos el valor de la cadena str --> ["fran", "juan", "hola"]

    def __len__(self):
        """Devuelve el número de elementos almacenados en el mapa"""
        assert len(self.str_to_id) == len(self.id_to_str)    # check
        return len(self.id_to_str)

    def _get_str(self, i):      # i --> id ya sea para term_id o doc_id
        """Devuelve la cadena correspondiente a un id numérico i dado.
           Si el índice no está presente lanza una excepción IndexError.
        """
        return self.id_to_str[i]    # i --> posicion en la posting list

    def _get_id(self, s):
        """Devuelve el identificador correspondiente a la cadena s.
           Si s no está en el mapa aún, se le asigna un nuevo id y es este valor.
        """
        idx = self.str_to_id.get(s, len(self.id_to_str))      # s --> cadena, le pasamos como segundo arg el indice siguiente para entrar en la lista (RECUERDA que se empieza en 0 y acabamos en n-1, len() da n)

        if s not in self.str_to_id:
            self.str_to_id[s] = idx
            self.id_to_str.append(s)

        assert len(self.str_to_id) == len(self.id_to_str)     # comprobacion de consistencia
        return idx

    def __getitem__(self, key):
        """Si key es un entero, usa _get_str;
           Si key es una cadena, usa _get_id;"""
        if type(key) is int:
            return self._get_str(key)
        elif type(key) is str:            # si testIdap["a"] --> como "a" es str --> se llama a _get_id("a")
            return self._get_id(key)
        else:
            raise TypeError

Asegúrate de que supera los siguientes tests sencillos:

In [ ]:
testIdMap = IdMap()
assert testIdMap['a'] == 0, "Imposible añadir una nueva cadena al IdMap"
assert testIdMap['bcd'] == 1, "Imposible añadir una nueva cadena al IdMap"
assert testIdMap['a'] == 0, "Imposible obtener el id de una cadena existente"
assert testIdMap[1] == 'bcd', "Imposible obtener la cadena correspondiente a un id dado"
try:
    testIdMap[2]
except IndexError as e:
    pass
else:
    print("No lanza correctamente una excepción IndexError al abrir un id fuera de rango")
assert len(testIdMap) == 2

# Construcción de un índice invertido en memoria: Algoritmo *naïve*

Ahora vamos a construir un índice invertido siguiendo un algoritmo *naïve* como el que se describe al comienzo del tema 2 de teoría, el cual no tiene en cuenta los requisitos de espacio en memoria del algoritmo ni las características del hardware del computador en el que se ejecuta. Según este algoritmo, el índice al completo, incluyendo tanto su diccionario de términos como las *postings lists* de cada término del vocabulario, se mantienen en memoria mientras se va construyendo el índice.

**IMPORTANTE: Llevar a cabo la construcción de esta forma resultará únicamente posible en el caso de que no estemos usando `ulimit` para imponer limitaciones el uso de memoria por parte del proceso encargado de ejecutar este *notebook* de Jupyter  (ver indicaciones al comienzo del cuaderno).**

Recordemos que, para construir un índice, los pasos generales son:
1. Escanear la colección reuniendo todos los pares término-documento.
2. Ordenar los pares con el término como clave primaria y el docID como clave secundaria.
3. Generar una lista con los docID de cada término

En nuestro caso, ya que no nos preocupará el tamaño de las estructuras de datos que mantengamos en memoria, realizaremos la construcción de una manera ligeramente distinta: en lugar de generar una lista de tuplas (termId,docId) durante el escaneo del corpus, utilizaremos un diccionario Python para mantener la correspondencia entre cada término (identificado por su *termId*) y la lista de documentos (*docIds*) en que aparece (*postings list*). El uso de estas estructuras de datos en memoria nos simplifica la tarea por varias razones:
- No es necesario ordenar por *termId* (orden primario de tuplas (termid, docid)), sino que los términos del diccionario están automáticamente ordenados por *termId* gracias a que, desde Python 3.7, los diccionarios son ordenados (mantienen el orden de inserción). Puesto que los términos se insertan en orden creciente de *termId*, a la hora de escribir el índice en disco podremos simplemente recorrer el diccionario a la hora de escribir las *postings lists* sin que sea necesario realizar ninguna ordenación previa.
- No es necesario ordenar por *docId* (orden secundario) sino que para cada término construiremos su *postings list* manteniendo el orden de los *docIds* en todo momento, al tiempo que evitamos duplicados. Esto es sencillo puesto que los documentos se escanean en orden creciente de *docId* (se generará un nuevo *docId* al considerar cada nuevo documento del corpus). Para cada término encontrado en el corpus, simplemente se comprueba si ya existe en el diccionario; en tal caso, basta con comparar el *docId* actual con el del último *posting* de la lista asociada el término, para decidir si resulta necesario añadirlo o no.

In [ ]:
# Posting list = docs donde apararece el temino "X" con id "Y"

In [ ]:
class NaiveIndex:
    """Una clase que implementa un índice invertido ingenuo (naive) construido enteramente en memoria.

    Attributes
    ---------
    term_id_map(IdMap): Mapea términos a termIDs
    doc_id_map(IdMap): Mapea rutas relativas de los documentos (p. ej. 0/3dradiology.stanford.edu_)
                       a docIDs
    data_dir(str): Ruta a los datos
    out_dir(str): Ruta al directorio donde los ficheros de índice se guardarán/leerán
    postings: Diccionario: termID->postings lists de la forma [docid1,docid2...]
    postings_metadata: Diccionario: termID->(start_position_in_index_file,
                                                    number_of_postings_in_list,
                                                    length_in_bytes_of_postings_list)
    Los metadatos de postings se usan para guardar en disco y recuperar las postings lists
    """
    def __init__(self, data_dir, out_dir):
        self.term_id_map = IdMap()    # esto es para llamar a la clase y poder usar sus funciones con .funcion
        self.doc_id_map = IdMap()
        self.data_dir = data_dir
        self.output_dir = out_dir
        self.postings = {}
        self.postings_metadata = {}

## *Parsear* los documentos del *corpus*

Programa los métodos `parse_all_subdirectories` y `parse_directory` de la clase `NaiveIndex`, de la forma que se indica:

- `parse_directory`: Escanea los documentos existentes en un directorio y los añade a un diccionario de *postings* que mantiene la asociación entre cada *termID* y su *postings list* (lista de *docIds*). Recibe como argumentos la ruta al directorio a escanear y el diccionario de *postings* sobre el que debe operar.

- `parse_all_subdirectories`: Debe hacer uso de la función anterior, para escanear todos los subdirectorios ('0', '1', etc.) en los que se dividen los ficheros del corpus. Como resultado de ejecutar esta función, el atributo `postings` de la clase `NaiveIndex` contendrá un diccionario con tantos elementos como términos distintos encontrados en el corpus, y para cada uno de ellos los *docId* en que aparece.

**ACLARACIÓN**: *En las siguientes celdas, el hacer que `NaiveIndex` herede de `NaiveIndex` no es más que una forma sencilla de añadir un nuevo método a una clase ya existente. Aunque esto puede resultar confuso, en realidad se utiliza simplemente para dividir en varias celdas una definición de clase dentro de un cuaderno de Jupyter.*

In [ ]:
class NaiveIndex(NaiveIndex):       # forma de añadir nuevos métodos en distintas celdas del notebook, evitando tener que repetir toda la clase cada vez
    def parse_directory(self, dir_path, postings):
        """Recorre todos los documentos en la ruta indicada y los inserta en el diccionario
        de posting lists dado como argumento. Los documentos deben ordenarse previamente para
        garantizar que nos da la misma solución.

        Se debe usar self.doc_id_map para obtener los docIDs y self.term_id_map para obtener los termIDs.
        Args:
            dir_path (str): Ruta al directorio que contiene los documentos a indexar
            postings (dict): Diccionario donde se insertarán las postings lists
        """
        # Lee las líneas de cada documento, extrae sus términos e inserta el docID en la
        # posting list de ese término
        ### Begin your code
        for filename in sorted(os.listdir(dir_path)):
            doc_path = os.path.join(dir_path, filename)   # construimos la ruta
            doc_id = self.doc_id_map[doc_path]    # Si el documento todavía no había aparecido, se le asignará automáticamente un nuevo docID

            with open(doc_path, 'r') as file:
                for line in file:
                    for term in line.split():
                        term_id = self.term_id_map[term]        # Como vimos anteriormente con IdMap, si el término ya existe recupera su ID y, si es nuevo, crea uno

                        # Creamos la lista para cada term_id en caso de que no exista
                        if term_id not in postings:
                            postings[term_id] = []

                        # Adjuntamos el doc_id a cada term_id (si es la primera vez) o evitar duplicados de docs asociados al mismo term_id
                        if not postings[term_id] or postings[term_id][-1] != doc_id:      # significa que la posting list está vacía --> Entonces necesariamente hay que añadir el primer documento O comprueba que el último documento almacenado no sea el documento que estamos procesando actualmente
                            postings[term_id].append(doc_id)
        ### End your code

    def parse_all_subdirectories(self):
        """Recorre todos los subdirectorios y genera el diccionario de posting lists. Los directorios deben ordenarse previamente.

        Se debe usar self.doc_id_map para obtener los docIDs y self.term_id_map para obtener los termIDs.
        """
        assert not self.postings      # Esto exige que el índice todavía esté vacío antes de iniciar el escaneo

        # Recuerda ordenar la lista de directorios para asegurarte de que se leen en el orden correcto
        ### Begin your code

        # recorre los distintos subdirectorios del corpus: Ex= directorio0 --> parse_directory(), directorio1 --> parse_directory(), ...
        for directory in sorted(os.listdir(self.data_dir)):
            dir_path = os.path.join(self.data_dir, directory)
            self.parse_directory(dir_path, self.postings)

        ### End your code

        # MUY IMPORTANTE: Al terminar, self.postings contiene el índice invertido completo en memoria

Utiliza los ficheros de juguete en la ruta indicada por `toy_dir` para probar tu código.

In [ ]:
!find $toy_dir -type f | sort

data/toy/0/fine.txt
data/toy/0/hello.txt
data/toy/1/byebye.txt
data/toy/1/bye.txt
data/toy/2/fine.txt
data/toy/2/hello.txt


Comprueba si la función funciona como se espera con los datos de juguete.

In [ ]:
with open(Path(toy_dir)/'0/fine.txt', 'r') as file:
    print(file.read())
with open(Path(toy_dir)/'0/hello.txt', 'r') as file:
    print(file.read())

i'm fine , thank you

hi hi
how are you ?



La siguiente celda comprueba si la función `parse_all_subdirectories` es capaz de escanear correctamente los ficheros de juguete situados en `toy_dir`, y a continuación, muestra el objeto índice creado con `obj.__dict__`, de forma que se pueda ver el valor del atributo `td_pairs`, entre otros.

Tras escanear los documentos y construir el diccionario sobre los datos de juguete, deberíamos tener un atributo `postings_lists` con el siguiente contenido:

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive_toy)     # creamos el índice
toy_index.parse_all_subdirectories()
toy_index.__dict__

{'term_id_map': <__main__.IdMap at 0x7d703cf6e490>,
 'doc_id_map': <__main__.IdMap at 0x7d703d39b890>,
 'data_dir': 'data/toy',
 'output_dir': 'output/naive/toy_index',
 'postings': {0: [0, 4],
  1: [0, 4],
  2: [0, 2, 4],
  3: [0, 4],
  4: [0, 1, 2, 4, 5],
  5: [1, 5],
  6: [1, 5],
  7: [1, 5],
  8: [1, 2, 5],
  9: [2],
  10: [2],
  11: [2],
  12: [2, 3],
  13: [2]},
 'postings_metadata': {}}

In [ ]:
toy_index.postings

{0: [0, 4],
 1: [0, 4],
 2: [0, 2, 4],
 3: [0, 4],
 4: [0, 1, 2, 4, 5],
 5: [1, 5],
 6: [1, 5],
 7: [1, 5],
 8: [1, 2, 5],
 9: [2],
 10: [2],
 11: [2],
 12: [2, 3],
 13: [2]}

Resultado:

```
{0: [0, 4],
 1: [0, 4],
 2: [0, 2, 4],
 3: [0, 4],
 4: [0, 1, 2, 4, 5],
 5: [1, 5],
 6: [1, 5],
 7: [1, 5],
 8: [1, 2, 5],
 9: [2],
 10: [2],
 11: [2],
 12: [2, 3],
 13: [2]}
```

Escribe algunas pruebas para asegurarte de que efectivamente el método `parse_all_subdirectories` funciona como se espera en los datos de juguete (`toy_dir`). Por ejemplo, deberías asegurarte de que una palabra dada recibe el mismo id cada vez que aparece).

In [ ]:

print("Term->TermIDs")
### Begin your code
for term, term_id in toy_index.term_id_map.str_to_id.items():
    print((term, term_id))
### End your code

print("Doc->DocIDs")
### Begin your code
for doc, doc_id in toy_index.doc_id_map.str_to_id.items():
    print((doc, doc_id))
### End your code

Term->TermIDs
("i'm", 0)
('fine', 1)
(',', 2)
('thank', 3)
('you', 4)
('hi', 5)
('how', 6)
('are', 7)
('?', 8)
('good', 9)
('to', 10)
('see', 11)
('bye', 12)
('later', 13)
Doc->DocIDs
('data/toy/0/fine.txt', 0)
('data/toy/0/hello.txt', 1)
('data/toy/1/bye.txt', 2)
('data/toy/1/byebye.txt', 3)
('data/toy/2/fine.txt', 4)
('data/toy/2/hello.txt', 5)


Resultado:
```
Term->TermIDs
("i'm", 0)
('fine', 1)
(',', 2)
('thank', 3)
('you', 4)
('hi', 5)
('how', 6)
('are', 7)
('?', 8)
('bye', 9)
('good', 10)
('to', 11)
('see', 12)
('later', 13)
Doc->DocIDs
('data/toy/0/fine.txt', 0)
('data/toy/0/hello.txt', 1)
('data/toy/1/byebye.txt', 2)
('data/toy/1/bye.txt', 3)
('data/toy/2/fine.txt', 4)
('data/toy/2/hello.txt', 5)
```

## Recuperación de información sobre el índice en memoria

Vamos a distinguir métodos que trabajan con la versión del índice que se mantiene en memoria y la versión que se conserva en disco, para que podamos comprobar la diferencia de procedimiento a la hora de acceder al índice en un caso y en el otro.

En estas celdas vamos a añadir métodos que trabajan con la version del índice que tenemos en memoria.

Ahora, añade a la clase `NaiveIndex` los siguientes métodos:

- `get_term_postings_from_mem`: Debe devolver la lista de *postings* (*docIds*) de un determinado término dado como argumento.

- `docids_to_docnames`: A partir de una lista de *postings* dada como argumento, debe devolver una lista en la que el elemento i-ésimo será la ruta al fichero correspondiente al *docId* i-esimo en la lista dada.

- `retrieve`: Debe devolver los nombres de los documentos en los que aparece un término dado como argumento.

In [ ]:
# "you"
#    ↓
# get_term_postings_from_mem
#    ↓
# [0, 1, 2, 4, 5]
#    ↓
# docids_to_docnames
#    ↓
# [rutas de los documentos]

In [ ]:
from collections import defaultdict

class NaiveIndex(NaiveIndex):

    def get_term_postings_from_mem(self, term):
        postings = []
        assert self.postings      # assert exige que exista un índice construido en memoria

        ### Begin your code
        term_id = self.term_id_map[term]        # convierte el término textual en su identificador
        postings = self.postings.get(term_id, [])     # recupera los docs_ids asociados al term_id
        ### End your code

        return postings

    def docids_to_docnames(self, postings):       # [1, 5]  es util pero el user realmente quiere el nombre del doc, no su id
        doc_names = []

        ### Begin your code
        doc_names = [self.doc_id_map[doc_id] for doc_id in postings]
        ### End your code

        return doc_names

    def retrieve(self, term):
        return self.docids_to_docnames(self.get_term_postings_from_mem(term))


Comprueba que tus métodos funcionan. Recuerda que, tras añadir nuevos métodos a una clase, debes volver a construir un nuevo objeto de dicha clase para que contenga los nuevos métodos añadidos por celdas como la anterior:

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive_toy)
toy_index.parse_all_subdirectories()

In [ ]:
toy_index.get_term_postings_from_mem("hi")

[1, 5]

Resultado:
```
[1, 5]
```

In [ ]:
toy_index.retrieve("you")

['data/toy/0/fine.txt',
 'data/toy/0/hello.txt',
 'data/toy/1/bye.txt',
 'data/toy/2/fine.txt',
 'data/toy/2/hello.txt']

Resultado:

```
['data/toy/0/fine.txt',
 'data/toy/0/hello.txt',
 'data/toy/1/bye.txt',
 'data/toy/2/fine.txt',
 'data/toy/2/hello.txt']
```

## Guardar el índice invertido en disco

Una vez construido el índice, queremos guardarlo en disco para poder utilizarlo posteriormente con el fin de recuperar información del mismo sin necesidad de volver a construirlo. Esto también nos permitirá liberar la memoria ocupada por las *postings lists*, que hasta este punto se mantienen en memoria (atributo `postings`). Los pasos a seguir son los siguientes:

1. Guardar en un fichero el índice, es decir, las *postings list* de todos los términos del vocabulario, en orden creciente por *termId*.

1. Guardar en un fichero los *metadatos* necesarios para localizar los *postings* de un término en el fichero de índice anterior.



### Escribir los `postings` a un fichero

Debes programar los métodos `save_term_postings` y `save_postings` de la clase `NaiveIndex`, como se indica a continuación:

- `save_term_postings`: A partir de un fichero `f` previamente abierto con `with open(...) as f`, debe obtener el *offset* actual y a continuación escribir la lista de *postings* dada como argumento. Para escribir  las *postings lists* (listas de *docIDs*) en el disco, utilizaremos el paquete `struct` y su método `pack`; el módulo `struct` contiene funciones estáticas de codificación y decodificación que permiten transformar una lista de enteros a un array de bytes, y viceversa. [Ver documentación aquí](https://docs.python.org/3/library/struct.html). En particular, se puede obtener una lista de enteros así: `struct.pack(f'{len(term_postings)}i', *term_postings)`

- `save_postings`: Debe crear un fichero en el directorio de salida del índice `output_dir`, con el nombre indicado por la variable `postings_index_filename`, en el que se guardará la *postings list* de cada término del vocabulario, en orden creciente por *termId*. Debe hacer uso de la función `save_term_postings` anterior. Conforme se van guardando las *posting lists* en el fichero, será necesario construir un nuevo diccionario (atributo `postings_metadata`) con la correspondencia entre cada término (*termId*) y los metadatos para localizar sus *postings* en el fichero del índice. Para cada término, los metadatos necesarios para recuperar posteriormente los postings son: el desplazamiento (*offset*) dentro del fichero de índice donde comienza, la longitud de la lista de *postings* y el tamaño en bytes de dicha lista.

#### Construir el atributo `postings_metadata` de la clase `NaiveIndex`
Se trata de un diccionario que relaciona los termIDs con una tripleta de metadatos que es útil para leer y escribir las *postings lists* en el archivo de índice a/desde el disco:

* `start_position_in_index_file`:posición (en bytes) de la *posting list* en el fichero índice.

* `number_of_postings_in_list`: número de entradas (docIDs) de lista.

* `length_in_bytes_of_postings_list`: longitud en bytes de la *postings list* codificada.

Se supone que este mapeo se mantiene en memoria cuando se trabaja con el índice.

In [ ]:
# Este bloque es importante porque marca el cambio entre dos etapas de la práctica:
# índice construido en memoria
#           ↓
# guardar postings en disco
#           ↓
# conservar metadatos para saber dónde está cada lista

# Hasta ahora self.postings contenía todas las posting lists. A partir de aquí empezamos a preparar el índice para que esas listas puedan mantenerse en disco y no sea necesario conservarlas permanentemente en memoria.


# ahora que queremos guardar el índice para:
  # - no tener que volver a construirlo cada vez;
  # - poder recuperar información posteriormente;
  # - poder liberar la memoria ocupada por las posting lists.


# El proceso se divide en dos partes:
  # 1. escribir todas las posting lists en un fichero;
  # 2. guardar metadatos que permitan saber dónde empieza y cuánto ocupa cada lista.
    # La idea conceptual va a ser:
    # postings.index
    # ──────────────────────────────────────────────
    # posting term 0 | posting term 1 | posting term 2 | ...


# Pero si metemos todas seguidas, surge una pregunta:
# ¿Cómo sabemos dónde empiezan y terminan las postings correspondientes a un término?

# Ahí entra postings_metadata.

# En memoria tenemos algo como:
# self.postings = {
#     0: [0, 4],
#     1: [0, 4],
#     2: [0, 2, 4]
# }

# Si hacemos self.postings[2] obtenenmos facilmente [0, 2, 4]
# El problema aparece cuando queremos quitar self.postings de memoria y guardarlo en disco.
# El fichero no guarda esos corchetes ni comas y guarda cada uno detras del otro. En realidad se escriben los enteros convertidos a bytes mediante struct.pack():
# 0  4  0  4  0  2  4

# Por eso hace falta el metadata para saber cuando empieza y termina cada uno, algo como:
# postings_metadata[0] = (0, 2, 8)   -->   El término 0 empieza en el byte 0, contiene 2 docIDs y ocupa 8 bytes (RECUERDA QUE CADA ENTERO OCUPA 4 BYTES)

# ¿Qué ocurriría después si quiero las postings del término 2? YA no necesitamos tener self.postings en memoria, consultamos self.postings_metadata[2], obtenemos (16, 3, 12) y con eso cogemos [0, 2, 4]

In [ ]:
import struct

class NaiveIndex(NaiveIndex):

    # SU mision es escribir la lista de docs_id y devolver donde empezço + numeros de bytes escritos
    def save_term_postings(self, file, term_postings):      # file es un fichero en modo binario
        """
        Escribe la lista de postings correspondientes a un término en un fichero
        OJO: No se puede usar pickle para guardar los postings, ya que queremos un formato binario compacto y eficiente. Usa la librería struct para empaquetar los enteros en binario.
        Args:
            file (file object): fichero abierto en modo binario para escritura
            term_postings (list): lista de docIDs (enteros) que forman la posting list
        """
        at_offset = -1          # desplazamiento en el disco donde se van a almacenar los postings (-1 es un valor inválido)
        num_bytes = 0
        assert term_postings, "save_term_postings espera una lista de postings no vacía"        # es algo como [0, 2, 4]

        ### Begin your code
        at_offset = file.tell()         # nos dice en qué posición del fichero estamos actualmente

        # Empaquetado
        packed_postings = struct.pack(
            f'{len(term_postings)}i',         # i es identificador del numero de docs a empaquetar
            *term_postings
        )
        num_bytes = file.write(packed_postings)     # escribe los bytes y devuelve cuántos se han escrito
        ### End your code

        assert at_offset >= 0 and num_bytes > 0, "save_term_postings no funciona correctamente."
        return at_offset, num_bytes     # nos proporciona justo la información necesaria para construir los metadatos

    def save_postings(self):
        """
        Escribe el índice invertido del objeto (self.postings_list) en un fichero
        """
        filename = postings_index_filename      # anteriormente definido como "postings.index"
        assert not self.postings_metadata, "Los metadatos de postings deben estar vacíos antes de guardar el índice en disco"

        ### Begin your code
        with open(os.path.join(self.output_dir, filename), 'wb') as file:         # w escritura, b binario --> necesario porque struct.pack() produce bytes
            for term_id in sorted(self.postings.keys()):        # guardar en orden creciente por term_id
                term_postings = self.postings[term_id]

                at_offset, num_bytes = self.save_term_postings(
                    file,
                    term_postings
                )

                self.postings_metadata[term_id] = (             # RESULTADO   0: (0, 2, 8)
                    at_offset,
                    len(term_postings),
                    num_bytes
                )
        ### End your code

        assert self.postings_metadata, "Los metadatos de postings no se han construido tras guardar el índice en el fichero"

    def release_postings_from_memory(self):
        """
        Libera los postings de memoria, manteniendo solamente los metadatos para poder localizar los postings en el fichero
        """
        self.postings = {}

    def release_metadata_from_memory(self):
        """
        Libera los metadatos de memoria
        """
        self.postings_metadata = {}

A continuación, prueba tu código y muestra `postings_metadata`:

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive_toy)
toy_index.parse_all_subdirectories()
toy_index.save_postings()

In [ ]:
toy_index.postings_metadata

{0: (0, 2, 8),
 1: (8, 2, 8),
 2: (16, 3, 12),
 3: (28, 2, 8),
 4: (36, 5, 20),
 5: (56, 2, 8),
 6: (64, 2, 8),
 7: (72, 2, 8),
 8: (80, 3, 12),
 9: (92, 1, 4),
 10: (96, 1, 4),
 11: (100, 1, 4),
 12: (104, 2, 8),
 13: (112, 1, 4)}

Resultado:
```
{0: (0, 2, 8),
 1: (8, 2, 8),
 2: (16, 3, 12),
 3: (28, 2, 8),
 4: (36, 5, 20),
 5: (56, 2, 8),
 6: (64, 2, 8),
 7: (72, 2, 8),
 8: (80, 3, 12),
 9: (92, 1, 4),
 10: (96, 1, 4),
 11: (100, 1, 4),
 12: (104, 2, 8),
 13: (112, 1, 4)}
```

Si tras guardar el índice en disco, volcamos con `hexdump` el contenido del fichero en el que hemos escrito los *postings*, deberíamos observar los *docIds* de cada *postings list*, uno tras otro, codificados como un entero de 4 bytes (en *little endian*, es decir, los bytes aparecerán al revés). NOTA: La primera columna mostrada por `hexdump` corresponde al offset del primer byte del fichero que aparece en esa línea (similar a como ocurría en el programa `okteta` que viste en la asignatura de "Fundamentos de Computadores"); el resto de columnas son el contenido del fichero.

In [ ]:
!ls $out_dir_naive_toy
!hexdump -C $out_dir_naive_toy/$postings_index_filename

docs.dict  postings.index  postings_metadata.dict  terms.dict
00000000  00 00 00 00 04 00 00 00  00 00 00 00 04 00 00 00  |................|
00000010  00 00 00 00 02 00 00 00  04 00 00 00 00 00 00 00  |................|
00000020  04 00 00 00 00 00 00 00  01 00 00 00 02 00 00 00  |................|
00000030  04 00 00 00 05 00 00 00  01 00 00 00 05 00 00 00  |................|
00000040  01 00 00 00 05 00 00 00  01 00 00 00 05 00 00 00  |................|
00000050  01 00 00 00 02 00 00 00  05 00 00 00 02 00 00 00  |................|
00000060  02 00 00 00 02 00 00 00  02 00 00 00 03 00 00 00  |................|
00000070  02 00 00 00                                       |....|
00000074


Resultado:

```
00000000  00 00 00 00 04 00 00 00  00 00 00 00 04 00 00 00  |................|
00000010  00 00 00 00 02 00 00 00  04 00 00 00 00 00 00 00  |................|
00000020  04 00 00 00 00 00 00 00  01 00 00 00 02 00 00 00  |................|
00000030  04 00 00 00 05 00 00 00  01 00 00 00 05 00 00 00  |................|
00000040  01 00 00 00 05 00 00 00  01 00 00 00 05 00 00 00  |................|
00000050  01 00 00 00 02 00 00 00  05 00 00 00 02 00 00 00  |................|
00000060  02 00 00 00 02 00 00 00  02 00 00 00 03 00 00 00  |................|
00000070  02 00 00 00                                       |....|
00000074
```

### Escribir los metadatos que referencian al fichero de *postings* (`postings_metadata`)

Debes programar el método `save_postings_metadata` de la clase `NaiveIndex`, que guarde en un fichero el contenido del diccionario `postings_metadata`. El nombre del fichero vendrá dado por la variable `postings_metadata_filename` y se creará igualmente en el directorio de salida; para cada término del vocabulario, se deberá escribir en orden creciente por *termId*, junto con el propio *termId*, la tripleta de metadatos anteriomente descrita *(offset, num_postings, num_bytes)*. NOTA: **No se debe asumir** que el diccionario `postings_metadata` está ordenado por *termId* (aunque en esta versión *naïve* en realidad sí lo está).

In [ ]:
# postings.index
#     → contiene los docID de las posting lists

# postings_metadata.dict
#     → contiene dónde está cada posting list dentro de postings.index

# Pero hay un problema: postings_metadata sigue siendo un diccionario Python en memoria.
# Si cerramos el programa, también lo perdemos.
# Así que ahora debemos guardar también este diccionario en disco.

In [ ]:
from collections import defaultdict

class NaiveIndex(NaiveIndex):

    def save_postings_metadata(self):
        """
        Escribe un diccionario de tripletas (offset, length, size_in_bytes) en un archivo binario.
        Se escriben los valores key, offset, length y size_in_bytes como enteros de 4 bytes, donde key es la clave
        del diccionario de tripletas.
        """
        filename = postings_metadata_filename
        if not self.postings_metadata:      # porque no tendría sentido intentar guardar unos metadatos que todavía no se han construido
            print("'save_postings_metadata' ha sido invocado pero existe una estructura de metadatos para guardar")
            return

        ### Begin your code

        # Los valores key, offset, length y size_in_bytes se escribirán como enteros de 4 bytes
        # ejemplo: struct.pack('i',key)
        with open(os.path.join(self.output_dir, filename), 'wb') as f:
            for key in sorted(self.postings_metadata.keys()):         # no debemos suponer que postings_metadata está ordenado por termID, aunque en esta implementación naïve haya terminado construyéndose así --> por eso forzamos con sorted()
                offset, length, size_in_bytes = self.postings_metadata[key]

                f.write(struct.pack('i', key))
                f.write(struct.pack('iii', offset, length, size_in_bytes))

        ### End your code

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive_toy)
toy_index.parse_all_subdirectories()
toy_index.save_postings()
toy_index.save_postings_metadata()

In [ ]:
toy_index.postings_metadata

{0: (0, 2, 8),
 1: (8, 2, 8),
 2: (16, 3, 12),
 3: (28, 2, 8),
 4: (36, 5, 20),
 5: (56, 2, 8),
 6: (64, 2, 8),
 7: (72, 2, 8),
 8: (80, 3, 12),
 9: (92, 1, 4),
 10: (96, 1, 4),
 11: (100, 1, 4),
 12: (104, 2, 8),
 13: (112, 1, 4)}

In [ ]:
!ls $out_dir_naive_toy
!hexdump -C $out_dir_naive_toy/$postings_metadata_filename

docs.dict  postings.index  postings_metadata.dict  terms.dict
00000000  00 00 00 00 00 00 00 00  02 00 00 00 08 00 00 00  |................|
00000010  01 00 00 00 08 00 00 00  02 00 00 00 08 00 00 00  |................|
00000020  02 00 00 00 10 00 00 00  03 00 00 00 0c 00 00 00  |................|
00000030  03 00 00 00 1c 00 00 00  02 00 00 00 08 00 00 00  |................|
00000040  04 00 00 00 24 00 00 00  05 00 00 00 14 00 00 00  |....$...........|
00000050  05 00 00 00 38 00 00 00  02 00 00 00 08 00 00 00  |....8...........|
00000060  06 00 00 00 40 00 00 00  02 00 00 00 08 00 00 00  |....@...........|
00000070  07 00 00 00 48 00 00 00  02 00 00 00 08 00 00 00  |....H...........|
00000080  08 00 00 00 50 00 00 00  03 00 00 00 0c 00 00 00  |....P...........|
00000090  09 00 00 00 5c 00 00 00  01 00 00 00 04 00 00 00  |....\...........|
000000a0  0a 00 00 00 60 00 00 00  01 00 00 00 04 00 00 00  |....`...........|
000000b0  0b 00 00 00 64 00 00 00  01 00 00 00 04 00 00 00  |....d...

Resultado:
```
00000000  00 00 00 00 00 00 00 00  02 00 00 00 08 00 00 00  |................|
00000010  01 00 00 00 08 00 00 00  02 00 00 00 08 00 00 00  |................|
00000020  02 00 00 00 10 00 00 00  03 00 00 00 0c 00 00 00  |................|
00000030  03 00 00 00 1c 00 00 00  02 00 00 00 08 00 00 00  |................|
00000040  04 00 00 00 24 00 00 00  05 00 00 00 14 00 00 00  |....$...........|
00000050  05 00 00 00 38 00 00 00  02 00 00 00 08 00 00 00  |....8...........|
00000060  06 00 00 00 40 00 00 00  02 00 00 00 08 00 00 00  |....@...........|
00000070  07 00 00 00 48 00 00 00  02 00 00 00 08 00 00 00  |....H...........|
00000080  08 00 00 00 50 00 00 00  03 00 00 00 0c 00 00 00  |....P...........|
00000090  09 00 00 00 5c 00 00 00  01 00 00 00 04 00 00 00  |....\...........|
000000a0  0a 00 00 00 60 00 00 00  01 00 00 00 04 00 00 00  |....`...........|
000000b0  0b 00 00 00 64 00 00 00  01 00 00 00 04 00 00 00  |....d...........|
000000c0  0c 00 00 00 68 00 00 00  02 00 00 00 08 00 00 00  |....h...........|
000000d0  0d 00 00 00 70 00 00 00  01 00 00 00 04 00 00 00  |....p...........|
000000e0
```

### Guardar el índice al completo en disco

Ahora, estamos en disposición de crear un método `save` que se encargue de guardar el índice construido al completo en disco, en cuatro sencillos pasos:

1. Guardar el mapeo entre términos y *termId* usando el método `save_terms_dict`.

1. Guardar el mapeo entre rutas a documentos y *docIds* usando el método `save_docs_dict`.

1. Guardar el fichero de índice con los *postings*.

1. Guardar el mapeo de los metadatos para localizar los *postings* en el fichero de índice.

1. Liberar la memoria ocupada por los *postings* y sus metadatos, una vez que ambos han sido escritos a disco.

Estos métodos os los damos hechos, pero esta vez usaremos el paquete Pickl que sirve para volcar a disco objetos Python. De esta manera podemos ver dos formas diferentes de generar un fichero con datos: uno creado por nosotros mismos a mano, y otro usando una solución más estándar. Sin embargo, hay que tener cuidado con Pickl porque tiene problemas de seguridad al poder ejecutar código malicioso al volver a cargar la estructura de datos del fichero.

In [ ]:
# Ahora ya sabemos guardar:
  # postings.index
  # postings_metadata.dict

# Pero para recuperar información posteriormente nos faltan otras dos piezas.
# Recordemos nuestros IdMap:
  # término ↔ termID
  # documento ↔ docID

# Sin ellos tendríamos las posting lists, pero no sabríamos qué termID corresponde a "you" ni qué ruta corresponde al docID 5.

# Por eso el índice completo estará compuesto por cuatro ficheros:
  # terms.dict
  #     "you" ↔ 4

  # docs.dict
  #     "data/toy/2/hello.txt" ↔ 5

  # postings.index
  #     ... docID 0, docID 1, docID 2 ...

  # postings_metadata.dict
  #     termID → dónde está su posting list

In [ ]:
class NaiveIndex(NaiveIndex):
    def save_terms_dict(self):
        with open(os.path.join(self.output_dir, terms_filename), 'wb') as f:        # En lugar de codificar manualmente estas estructuras mediante struct, el PDF utiliza pickle para guardar directamente los objetos Python IdMap
            pkl.dump(self.term_id_map, f)


#     El documento aprovecha esto para enseñarnos dos formas diferentes de almacenamiento:

  # postings / metadatos
  #        ↓
  # struct → formato construido manualmente


  # IdMap
  #        ↓
  # pickle → serialización de objetos Python  --> problema de seguridad de pickle: al cargar datos puede ejecutar código malicioso, por lo que hay que tener cuidado con este formato


    def save_docs_dict(self):
        with open(os.path.join(self.output_dir, docs_filename), 'wb') as f:
            pkl.dump(self.doc_id_map, f)

    def save(self):

        """
        Guarda el índice completo en disco.

        Vuelca doc_id_map y term_id_map en el directorio destino del índice en disco. Recuerda que
        al comenzar el notebook definimos las siguientes variables globales:
            terms_filename = 'terms.dict'
            docs_filename = 'docs.dict'
            postings_index_filename = 'postings.index'
            postings_metadata_filename = 'postings_metadata.dict'

        Estas variables nos dan el nombre de fichero donde se va a guardar cada diccionario y el índice invertido.
        """
        ### Begin your code
        self.save_terms_dict()
        self.save_docs_dict()
        self.save_postings()
        self.save_postings_metadata()

        # self.release_postings_from_memory()
        # self.release_metadata_from_memory()
        ### End your code

In [ ]:
# 1. save_terms_dict()
#        ↓
#    terms.dict

# 2. save_docs_dict()
#        ↓
#    docs.dict

# 3. save_postings()
#        ↓
#    postings.index
#    +
#    construye self.postings_metadata

# 4. save_postings_metadata()
#        ↓
#    postings_metadata.dict

# 5. liberar postings y metadatos de memoria

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive_toy)
toy_index.parse_all_subdirectories()
toy_index.save()

In [ ]:
# Más adelante veremos justamente el proceso contrario:
# postings_metadata.dict
#         ↓
# load_postings_metadata()
#         ↓
# self.postings_metadata

## Cargar los metadatos del índice en disco para poder recuperar información

A partir de este momento, el fichero de *postings* se mantiene en disco, y sólo es necesario mantener en memoria los metadatos necesarios para poder acceder a las partes del fichero de *postings* donde se guardan los *docIds* en los que aparecen los términos de la consulta. Así pues, para poder realizar recuperación de información sobre el índice en disco, debes programar los siguientes métodos de `NaiveIndex`:

- `load_next_metadata_entry`: Lee una entrada del fichero de metadatos, y devuelve una cuádrupla con el *termId* y los metadatos para leer la *posting list* del fichero de índice. La cuádrupla estará  formada por *(termId, offset, número de *postings*, tamaño en bytes de la lista)*.

- `load_postings_metadata`: Carga el fichero de metadatos de disco, haciendo uso de la función `load_next_metadata_entry` y construye el atributo `postings_metadata` de la clase `NaiveIndex`.

- `load`: Método que utilizará el código cliente que haga uso de `NaiveIndex` para cargar un índice y poder utilizarlo para realizar consultas. Cargará en memoria tanto el diccionario (mapeo de términos con *termIds* y documentos con *docIds*) como los metadatos.

In [ ]:
# Hasta aquí:
  # CORPUS
  #  ↓
  # parse
  #  ↓
  # índice en memoria
  #  ↓
  # save
  #  ↓
  # ÍNDICE EN DISCO

# Ahora queremos:
  # ÍNDICE EN DISCO
  #  ↓
  # load
  #  ↓
  # poder hacer consultas

# Pero no queremos cargar todas las posting lists en memoria otra vez.
# vamos a mantener en memoria solamente:
  # term_id_map
  # doc_id_map
  # postings_metadata

# mientras:
  # postings.index

# permanece en disco.

# Cuando necesitemos un término concreto, utilizaremos sus metadatos para ir únicamente a su trozo de postings.index.

In [ ]:
class NaiveIndex(NaiveIndex):
    def load_terms_dict(self):
        ## Load term<->termID mapping
        with open(os.path.join(self.output_dir, terms_filename), 'rb') as f:
            self.term_id_map = pkl.load(f)
    def load_docs_dict(self):
        ## Load doc<->docID mapping
        with open(os.path.join(self.output_dir, docs_filename), 'rb') as f:
            self.doc_id_map = pkl.load(f)

    def load_next_metadata_entry(self, f):          # Relación escritura ↔ lectura
        """ Lee la siguiente entrada de metadatos del fichero abierto f y devuelve una tupla
            (key, offset, length, size_in_bytes) o (None,None,None,None) si no hay más entradas.

            key es la clave del diccionario de metadatos (termID).
            offset es la posición en bytes en el fichero de índice donde comienza la posting list.
            length es el número de docIDs en la posting list.
            size_in_bytes es el tamaño en bytes de la posting list.
            """
        key_data = None
        key_data = f.read(4)        # ¿como detectamos le final? Mientras haya cuatro bytes disponibles, existe otra entrada --> final:(None, None, None, None)
        if key_data:
            key = struct.unpack('i', key_data)[0]       # unpack devuelve una estructura con el entero leído, y [0] extrae ese valor
            offset, length, size_in_bytes = struct.unpack('iii', f.read(12))          # recupera de golpe los tres valores restantes
            return key, offset, length, size_in_bytes
        return None,None,None,None

    def load_postings_metadata(self):
        """ Carga los metadatos de los postings desde su fichero """
        filename = postings_metadata_filename
        assert not self.postings_metadata

        with open(os.path.join(self.output_dir, filename), 'rb') as f:
            while True:
                key, offset, length, size_in_bytes = \
                    self.load_next_metadata_entry(f)

                if key is None:
                    break

                self.postings_metadata[key] = (
                    offset,
                    length,
                    size_in_bytes
                  )
        ### End your code

    def load(self):
        """ Carga el índice a partir del directorio de salida, donde se encuentra almacenado.
            Ten en cuenta que, como va a funcionar desde disco, solamente tendremos que tener en memoria
            los diccionarios de términos y documentos, y los metadatos de postings para poder
            localizar las postings lists en el fichero de índice.
        """
        assert not self.postings_metadata

        ### Begin your code
        self.load_terms_dict()
        self.load_docs_dict()
        self.load_postings_metadata()
        ### End your code

## Recuperación de información a partir del índice en disco

Para poder utilizar el índice invertido directamente desde disco, sin tener que cargarlo totalmente en memoria, lo que se hace es mantener los diccionarios de términos y documentos en memoria (supondremos que eso sí que nos cabe) y las posting lists en el disco, por lo que cuando se quiera consultar los documentos donde aparece un término, se usarán los diccionarios en memoria para localizar el termID, y a partir de los metadatos, ir a disco y leer la lista de postings para devolverla.

Para ello, debes programar los siguientes métodos de la clase `NaiveIndex`, necesarios para leer de disco el diccionario y poder realizar recuperación de información a partir del índice invertido:

- `load_posting_at_file_offset`: Debes desplazarte al `offset` del fichero `f` que se pasa como parámetro y a partir de ahí leer `length` enteros de 4 bytes. Usa `struct.unpack` para extraer la lista de los enteros leída.

- `get_term_postings_from_file`: Debe asumir que los `postings_metadata` están ubicados en memoria, y a partir del mapeo de términos a *termId*, debe acceder al desplazamiento adecuado del fichero del índice y leer la *posting list* del término, haciendo uso de la función `load_next_posting` anterior.

- `retrieve`: Es la función central de recuperación de información mediante el índice. Debe comprobar si los *postings* están ubicados en memoria (atributo `postings`) o en disco, en cuyo caso usará `get_term_postings_from_file` para leer únicamente de disco los *postings* del término consultado. En caso de que el índice esté íntegramente cargado en memoria, evitará acceder a disco. Adicionalmente, recibirá un argumento `sanity_check`, en función del cual el método comprobará que `get_term_postings_from_file` devuelve lo mismo que `get_term_postings_from_mem`, siempre que tanto `postings` como `postings_metadata` estén adecuadamente construidos.

In [ ]:
# HASTA AHORA:

# "you"
#    ↓
# term_id_map
#    ↓
# termID = 4
#    ↓
# postings_metadata[4]
#    ↓
# (offset, length, size_in_bytes)
#    ↓
# postings.index
#    ↓
# leer SOLO los bytes de esa posting list
#    ↓
# [0, 1, 2, 4, 5]
#    ↓
# doc_id_map
#    ↓
# rutas de los documentos

# AHORA TENEMOS TRES MÉTODOS NUEVOS:
# load_posting_at_file_offset
#     "Lee bytes concretos del fichero"

# get_term_postings_from_file
#     "Averigua qué bytes corresponden al término"

# retrieve
#     "Decide si recuperar desde memoria o desde disco"

In [ ]:
class NaiveIndex(NaiveIndex):

    def load_posting_at_file_offset(self, f, offset, length):
        assert length > 0
        postings_list = None

        ### Begin your code
        f.seek(offset)        # coloca el cursor justo donde comienza la posting list del termID offset
        postings_data = f.read(length * 4)
        postings_list = list(
            struct.unpack(f'{length}i', postings_data)      # bytes → enteros
        )
        ### End your code

        return postings_list

    def get_term_postings_from_file(self, term):      # recibe "you" y debe localizar auto donde está su posting list en disco
        disk_postings = []
        assert self.postings_metadata, "Debe leerse el diccionario de metadatos antes de poder acceder al índice"

        ### Begin your code
        term_id = self.term_id_map[term]

        offset, length, size_in_bytes = \
            self.postings_metadata[term_id]           # 4: (36, 5, 20)

        filename = postings_index_filename          # ahora abrimos postings.index

        with open(os.path.join(self.output_dir, filename), 'rb') as f:
            disk_postings = self.load_posting_at_file_offset(
                f,
                offset,
                length
            )
        ### End your code

        return disk_postings


# Antes consultar "you" implicaba self.postings[4]

# Ahora hacemos:
  # "you"
  #  ↓
  # 4
  #  ↓
  # (36, 5, 20)
  #  ↓
  # postings.index
  #  ↓
  # solo bytes 36–55


    # Hasta ahora retrieve() solo sabía consultar la versión en memoria, ahora debe poder funcionar en ambos escenarios:
    def retrieve(self, term, sanity_check=False):       # sanity_check --> Este parámetro está pensado para comprobar que hemos guardado y recuperado correctamente el índice.
        postings = []
        sanity_postings = None
        # Si sanity_check=True, se debe comprobar si hay una versión del índice cargada en memoria (el atribugo postings no estará vacío) y en ese caso, comparar la leída de disco y la que está en memoria, que deben ser la misma

        ### Begin your code
        if self.postings:           # si existe en memoria --> Sí → usa get_term_postings_from_mem() asi no accede a disco innecesariamente
            postings = self.get_term_postings_from_mem(term)

            if sanity_check and self.postings_metadata:         # sino accede a disco
                sanity_postings = self.get_term_postings_from_file(term)
                assert postings == sanity_postings            # Si coinciden, tenemos una comprobación de que lo escrito en disco se está recuperando correctamente

        else:
            postings = self.get_term_postings_from_file(term)

        ### End your code

        doc_names = [self.doc_id_map[i] for i in postings]
        return doc_names

In [ ]:
toy_index = NaiveIndex(toy_dir, out_dir_naive)
toy_index.parse_all_subdirectories()
toy_index.save()
toy_index.retrieve("you", sanity_check=True)

['data/toy/0/fine.txt',
 'data/toy/0/hello.txt',
 'data/toy/1/bye.txt',
 'data/toy/2/fine.txt',
 'data/toy/2/hello.txt']

In [ ]:
!grep -rlw "you" $toy_dir | sort

data/toy/0/fine.txt
data/toy/0/hello.txt
data/toy/1/bye.txt
data/toy/2/fine.txt
data/toy/2/hello.txt


## Indexando el corpus de documentos íntegro de forma *naïve* provoca `MemoryError`

Una vez hayas realizado un conjunto suficiente de pruebas sobre el conjunto de datos de juguete, es posible indexar el corpus al completo. Sin embargo, si estás aplicando los límites en el uso de memoria indicados, la celda siguiente mostrará un fallo `MemoryError` durante la ejecución, ya que generar el diccionario de términos con sus postings lists requiere más memoria de la disponible.

> Si deseas completar el indexado usando el algoritmo *naive*, debes establecerlo en la variable `build_naive_index_full` al comienzo de este *notebook* y luego ejecutarlo con `jupyter-notebook` desde otro terminal del shell en el que **no hayas impuesto mediante `ulimit` ningún límite en el uso de la memoria**.

In [ ]:
# ¿Dónde está el problema de memoria?
# Es importante distinguir construcción y consulta.
# Durante la construcción naïve
# Tenemos:
  # corpus completo
  #       ↓
  # parse_all_subdirectories()
  #       ↓
  # self.postings completo en RAM

# Ahí está el problema.
# Después de guardarlo
# Podemos trabajar así:

  # RAM
  # term_id_map
  # doc_id_map
  # postings_metadata

  # DISCO
  # postings.index

# y entonces una consulta no necesita cargar todo el índice.
# Por tanto, guardar el índice en disco nos ayuda durante su uso posterior, pero el algoritmo naïve todavía necesita memoria suficiente para construirlo inicialmente.

# Esta distinción es fundamental en esta práctica.

In [ ]:
build_naive_index_full = True

In [ ]:
# Vamos a crear un índice completo en memoria a partir del corpus de documentos y a guardarlo en disco
if build_naive_index_full:
    naive_index_full = NaiveIndex(corpus_dir, out_dir_naive)
    naive_index_full.parse_all_subdirectories()
    naive_index_full.save()

In [ ]:
!ls -al $out_dir_naive

total 74156
drwxr-xr-x 2 root root     4096 Oct  9 20:56 .
drwxr-xr-x 4 root root     4096 Oct  9 17:24 ..
-rw-r--r-- 1 root root  7992679 Oct  9 22:15 docs.dict
-rw-r--r-- 1 root root 55286272 Oct  9 22:15 postings.index
-rw-r--r-- 1 root root  5553136 Oct  9 22:15 postings_metadata.dict
-rw-r--r-- 1 root root  7088518 Oct  9 22:15 terms.dict


Ahora probemos a usar el índice para recuperar todos los documentos que contienen el término "porcupine" (puercoespín). Usamos sanity_check para comprobar que, aunque busca la lista de postings en el disco, la compara con la que hay en memoria para confirmar que es la misma, y que, por tanto, lo que hay en el disco se grabó correctamente y lo estamos recuperando bien.

In [ ]:
if build_naive_index_full:
    naive_index_full.retrieve("porcupine", sanity_check=True)

In [ ]:
!grep -rlw "porcupine" $corpus_dir | sort

data/corpus/5/searchworks.stanford.edu_view_5614345
data/corpus/6/wnt.stanford.edu_
data/corpus/7/www.scs.stanford.edu_07wi-cs244b_notes_
data/corpus/9/www.stanford.edu_group_dahlia_genetics_cultivars_valley_porcupine_valley_porcupine.htm
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_inhibitors
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_porcupine
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_smallmolecules
data/corpus/9/www.stanford.edu_group_Urchin_language.htm


El resultado debería ser el siguiente:

```data/corpus/5/searchworks.stanford.edu_view_5614345
data/corpus/6/wnt.stanford.edu_
data/corpus/7/www.scs.stanford.edu_07wi-cs244b_notes_
data/corpus/9/www.stanford.edu_group_dahlia_genetics_cultivars_valley_porcupine_valley_porcupine.htm
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_inhibitors
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_porcupine
data/corpus/9/www.stanford.edu_group_nusselab_cgi-bin_wnt_smallmolecules
data/corpus/9/www.stanford.edu_group_Urchin_language.htm````

## Cargar índice desde disco

Por supuesto, una vez que el índice ya se encuentra en disco codificado adecuadamente en los ficheros .dict y .index, podemos acceder diréctamente a él simplemente creando un objeto índice y usando el método load() que cargará los metadatos y los diccionarios de términos y documentos. Para hacer la prueba, vamos a crear un índice nuevo a partir de los ficheros en disco, y vamos a comprobar que lo que se lee de él coincide con la versión que tenemos en memoria si es que hay alguna. En este caso no tiene por qué haber una versión en memoria al ser un nuevo índice leído del disco.

In [ ]:
if build_naive_index_full:
    index_in_disk = NaiveIndex(corpus_dir, out_dir_naive)       # ¡FIJATE! Esto es un objeto nuevo, NO HACEMOS index_in_disk.parse_all_subdirectories() --> este nuevo objeto NO vuelve a indexar el corpus. En su lugar, lee el indice que habiamos guardado previamente
    index_in_disk.load()
    index_in_disk.retrieve("porcupine")
    index_in_disk.retrieve("porcupine", sanity_check=True)

In [ ]:
# Eso significa:
# PRIMERA VEZ

# corpus
#  ↓
# parse_all_subdirectories()
#  ↓
# construcción costosa
#  ↓
# save()
#  ↓
# índice en disco


# Después, en otra utilización:
# índice ya existente en disco
#  ↓
# load()
#  ↓
# retrieve("porcupine")

# No vuelves a construir el índice.

# Es decir, para crear el indice hace falta

In [ ]:
# Por tanto, build_naive_index_full:

  # False → desarrollo y pruebas con toy_dir para probar que todo funciona y el índice funciona
  # True  → ejecutar finalmente el indexado completo sobre corpus_dir